In [ ]:
!pip -q install transformers accelerate sentencepiece pandas tqdm tenacity

import json, re, math
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
from tqdm import tqdm

In [ ]:
DATA_PATH = "/content/gsm_variation_100base.jsonl"


rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

print(df.columns.tolist())
print(df[["split", "split2", "variation_type"]].head())

['base_id', 'variant_id', 'family', 'source', 'config', 'split', 'deduped_index', 'question', 'final_answer', 'family_params', 'gen_applied', 'gen_notes', 'dataset', 'id', 'variation_type', 'split2', 'answer']
   split split2      variation_type
0  train   base                base
1  train    var  irrelevant_numeric
2  train    var       symbolic_swap
3  train    var         scale_tweak
4  train    var            rephrase


In [ ]:
df = df.rename(columns={"split": "split_task"})

df["split"] = df["split2"].astype(str).str.strip().str.lower()

print(df["split"].value_counts().to_dict())
print(df["variation_type"].value_counts().to_dict())
df = df.drop(columns=["split2"])

# Sanity checks
print("Split counts:", df["split"].value_counts().to_dict())
assert df["split"].value_counts().to_dict().get("base", 0) == 100
assert df["split"].value_counts().to_dict().get("var", 0) == 500

{'var': 500, 'base': 100}
{'base': 100, 'irrelevant_numeric': 100, 'symbolic_swap': 100, 'scale_tweak': 100, 'rephrase': 100, 'critical_thinking': 100}
Split counts: {'var': 500, 'base': 100}


In [ ]:
pilot_rows = []
pilot_rows.append(df[df["split"]=="base"].sample(1, random_state=0))
for vt in sorted(df[df["split"]=="var"]["variation_type"].unique()):
    pilot_rows.append(df[(df["split"]=="var") & (df["variation_type"]==vt)].sample(1, random_state=0))
pilot_df = pd.concat(pilot_rows, ignore_index=True)
pilot_df[["id","split","variation_type","base_id"]]

,id,split,variation_type,base_id
0,gsm100_0027__clean__v1,base,base,gsm100_0027
1,gsm100_0027__implicit_reasoning_required__v1,var,critical_thinking,gsm100_0027
2,gsm100_0027__irrelevant_numeric_distractor__v1,var,irrelevant_numeric,gsm100_0027
3,gsm100_0027__semantic_rephrasing__v1,var,rephrase,gsm100_0027
4,gsm100_0027__scale_or_unit_change__v1,var,scale_tweak,gsm100_0027
5,gsm100_0027__symbolic_or_operand_swap__v1,var,symbolic_swap,gsm100_0027


In [ ]:
HARD_VARIATIONS = {"symbolic_swap", "scale_tweak", "irrelevant_numeric"}

In [ ]:
df_sc = df[(df["split"] == "base") | (df["variation_type"].isin(HARD_VARIATIONS))].copy()
print("SC subset size:", len(df_sc))
print(df_sc["variation_type"].value_counts())

SC subset size: 400
variation_type
base                  100
irrelevant_numeric    100
symbolic_swap         100
scale_tweak           100
Name: count, dtype: int64


In [ ]:
SYSTEM = (
    "Solve the math problem carefully.\n"
    "Briefly explain with 1-3 sentences.\n"
    "End with exactly one line in the format:\n"
    "FINAL: <answer>\n"
)

def build_prompt(question: str):
    return [{"role":"system","content": SYSTEM},
            {"role":"user","content": question}]

# Strict FINAL line
FINAL_RE = re.compile(r"^\s*FINAL\s*:\s*(.+?)\s*$", re.IGNORECASE)

# Time-like token: 6:00 pm, 6 pm, 18:00, 6:15am
TIME_RE = re.compile(r"\b(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\b", re.IGNORECASE)


NUM_RE = re.compile(
    r"(?<![A-Za-z0-9_])"                       # no letter/digit right before
    r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)"        # 1,234 or 1234
    r"(?:\.\d+)?"                             # optional decimal
    r"(?:[eE][-+]?\d+)?"                      # optional scientific notation
    r"(?![A-Za-z_])"                          # no letter right after
)

# Useful fallback cues near the end
ANSWER_CUE_RE = re.compile(
    r"(answer|therefore|thus|so|hence|result|x\s*=)\s*[:=]?\s*"
    r"([-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)",
    re.IGNORECASE
)

def extract_final(text: str) -> str:
    """
    Robust extraction:
    1) last line starting with FINAL:
    2) answer-like cue near the end
    3) last standalone number in the last few lines
    4) final fallback: last standalone number anywhere
    """
    if not text:
        return ""

    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]

    # 1) Prefer the LAST line that starts with FINAL:
    for ln in reversed(lines):
        m = FINAL_RE.match(ln)
        if m:
            return m.group(1).strip()

    # 2) Search answer-like cues in the tail only
    tail = "\n".join(lines[-8:])  # last few lines
    cue_matches = list(ANSWER_CUE_RE.finditer(tail))
    if cue_matches:
        return cue_matches[-1].group(2).strip()

    # 3) Search last few lines for a standalone number
    for ln in reversed(lines[-8:]):
        nums = NUM_RE.findall(ln)
        if nums:
            return nums[-1].strip()

    # 4) Fallback: last standalone number anywhere
    nums = NUM_RE.findall(text)
    if nums:
        return nums[-1].strip()

    # Last non-empty line
    return lines[-1] if lines else ""

def normalize_answer(ans: str) -> str:
    """
    Numeric normalization:
    - keeps only the numeric value
    - supports comma-formatted numbers
    - maps time like 6:00 pm -> 6
    - strips $ and commas automatically
    """
    if ans is None:
        return ""

    t = str(ans).strip().lower()
    if not t:
        return ""

    # Remove obvious wrappers
    t = t.replace("$", "").strip()

    # Time handling first
    tm = TIME_RE.search(t)
    if tm and (":" in tm.group(0) or tm.group(3) in ("am", "pm")):
        hour = int(tm.group(1))
        return str(hour)

    # Extract last standalone number
    nums = NUM_RE.findall(t)
    if not nums:
        return ""

    try:
        x = float(nums[-1].replace(",", ""))
        if abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        return str(x)
    except:
        return nums[-1].replace(",", "")

def is_correct(pred, gold, tol=1e-6) -> bool:
    p = normalize_answer(pred)
    g = normalize_answer(gold)
    if p == "" or g == "":
        return False

    try:
        pf = float(p)
        gf = float(g)
        return abs(pf - gf) <= tol
    except:
        return p == g

In [ ]:
from dataclasses import dataclass
from typing import Any
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

@dataclass
class HFChatBackend:
    model_id: str
    use_pipeline: bool = False
    torch_dtype: Any = torch.float16

    def __post_init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            device_map="auto",
            torch_dtype=self.torch_dtype,
            trust_remote_code=True,
        )
        self.pipe = None  # keep attribute so other code doesn't break

        # Pad token safety (prevents warnings / generation errors)
        if self.tokenizer.pad_token_id is None and self.tokenizer.eos_token_id is not None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    @torch.inference_mode()
    def generate(self, messages, max_new_tokens=128, temperature=0.2, top_p=0.95) -> str:
        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device)

        do_sample = (temperature is not None) and (temperature > 0)

        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        if do_sample:
            gen_kwargs.update(dict(temperature=temperature, top_p=top_p))

        outputs = self.model.generate(**inputs, **gen_kwargs)

        return self.tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import login
login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 129.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
deepseek_r1_qwen7b = HFChatBackend("deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", use_pipeline=False)
qwen3_4b = HFChatBackend("Qwen/Qwen3-4B", use_pipeline=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
qwen3_06b = HFChatBackend("Qwen/Qwen3-0.6B", use_pipeline=False)
qwen3_17b = HFChatBackend("Qwen/Qwen3-1.7B", use_pipeline=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
openthoughts = HFChatBackend("open-thoughts/OpenThinker-Agent-v1", use_pipeline=False)
qwen25_7b    = HFChatBackend("Qwen/Qwen2.5-7B-Instruct", use_pipeline=False)
llama32_3b = HFChatBackend("meta-llama/Llama-3.2-3B-Instruct", use_pipeline=False)

In [ ]:
def run_self_consistency(
    backend,
    messages,
    n=3,
    max_new_tokens=128,
    temperature=0.8,
    top_p=0.95,
    keep_raw=False,
    early_stop=True,
    max_n_if_tied=5,   # NEW: expand beyond n if needed
):
    """
    Returns:
      voted: chosen normalized answer string
      counts: dict of normalized answer -> vote count
      raw: list of raw generations (if keep_raw=True, else [])
    """
    raw = [] if keep_raw else None
    counts = {}
    sample_info = []   # stores (norm_pred, raw_text)

    target_n = n
    i = 0

    while i < target_n:
        txt = backend.generate(
            messages,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p
        )

        final = extract_final(txt)
        norm = normalize_answer(final)

        counts[norm] = counts.get(norm, 0) + 1
        sample_info.append((norm, txt))

        if keep_raw:
            raw.append(txt)

        i += 1

        # majority threshold for current target_n
        need = (target_n // 2) + 1

        # early stop if strict majority reached
        if early_stop and counts[norm] >= need:
            break

        # if we've finished n samples and there is a tie, expand up to max_n_if_tied
        if i == target_n:
            max_count = max(counts.values()) if counts else 0
            num_with_max = sum(1 for v in counts.values() if v == max_count)

            if num_with_max > 1 and target_n < max_n_if_tied:
                target_n += 1  # sample one more to break tie

    # Final vote
    if counts:
        max_count = max(counts.values())
        winners = [k for k, v in counts.items() if v == max_count]

        if len(winners) == 1:
            voted = winners[0]
        else:
            # deterministic numeric fallback: choose candidate closest to median if numeric
            numeric_winners = []
            for w in winners:
                try:
                    numeric_winners.append((w, float(w)))
                except:
                    pass

            if numeric_winners:
                vals = sorted(v for _, v in numeric_winners)
                mid = vals[len(vals)//2]
                voted = min(numeric_winners, key=lambda x: abs(x[1] - mid))[0]
            else:
                # fallback: choose the last winner seen
                voted = None
                for norm, _ in reversed(sample_info):
                    if norm in winners:
                        voted = norm
                        break
                if voted is None:
                    voted = winners[0]
    else:
        voted = ""

    return voted, counts, (raw if keep_raw else [])

In [ ]:
def evaluate_simple(
    backend,
    model_name: str,
    df: pd.DataFrame,
    method: str = "cot",  # "cot" or "sc"
    sc_n: int = 3,
    limit: Optional[int] = None,
    seed: int = 0,
    max_new_tokens: int = 128,
    temperature_cot: float = 0.2,
    temperature_sc: float = 0.8,
    top_p: float = 0.95,
    keep_raw: bool = True,  # now means: keep raw ONLY for wrong answers (fast + small)
) -> pd.DataFrame:

    d = df.copy()
    if limit is not None:
        d = d.sample(n=min(limit, len(d)), random_state=seed).reset_index(drop=True)

    recs = []
    for _, row in tqdm(d.iterrows(), total=len(d)):
        msgs = build_prompt(row["question"])

        raw_text = None
        vote_counts = None

        if method == "sc":
            # Keep raw samples so we can align rationale with the voted answer
            voted, counts, raw_samples = run_self_consistency(
                backend, msgs, n=sc_n,
                max_new_tokens=max_new_tokens,
                temperature=temperature_sc, top_p=top_p,
                keep_raw=True, early_stop=True, max_n_if_tied=5
            )
            pred = voted
            vote_counts = counts
            ok = is_correct(pred, row["answer"])

            # Save rationale only for wrong answers, but make it match the voted pred if possible
            if keep_raw and (not ok):
                matched = None
                for raw_candidate in raw_samples:
                    if normalize_answer(extract_final(raw_candidate)) == normalize_answer(pred):
                        matched = raw_candidate
                        break
                raw_text = matched if matched is not None else (raw_samples[0] if raw_samples else None)

        else:
            # CoT: run once
            raw = backend.generate(
                msgs, max_new_tokens=max_new_tokens,
                temperature=temperature_cot, top_p=top_p
            )
            pred = extract_final(raw)
            ok = is_correct(pred, row["answer"])

            # Save rationale ONLY if wrong
            if keep_raw and (not ok):
                raw_text = raw

        recs.append({
            "dataset": row.get("dataset", "gsm"),
            "split": row["split"],
            "variation_type": row["variation_type"],
            "base_id": row["base_id"],
            "id": row["id"],
            "question": row["question"],
            "gold": row["answer"],
            "pred": pred,
            "correct": bool(ok),
            "raw_response": raw_text,  # only filled for wrong cases now
            "model": model_name,
            "method": method,
            "sc_n": sc_n if method == "sc" else 1,
            "vote_counts": vote_counts,
        })

    return pd.DataFrame(recs)

In [ ]:
def safe_pdr(acc_base: float, acc_var: float) -> float:
    if acc_base is None or (isinstance(acc_base, float) and math.isnan(acc_base)):
        return float("nan")
    if acc_base <= 1e-12:
        return float("nan")
    return (acc_base - acc_var) / acc_base * 100.0

def summarize_pdr_clean(run_df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (model, method), g in run_df.groupby(["model","method"]):
        base = g[g["split"]=="base"]
        var  = g[g["split"]=="var"]

        acc_base = base["correct"].mean() if len(base) else float("nan")
        acc_var_all = var["correct"].mean() if len(var) else float("nan")

        out.append({
            "model": model,
            "method": method,
            "acc_base": acc_base,
            "acc_var_all": acc_var_all,
            "pdr_all_%": safe_pdr(acc_base, acc_var_all),
            "n_base": int(len(base)),
            "n_var": int(len(var)),
            "variation_type": "__ALL__",
            "acc_var": acc_var_all,
            "pdr_%": safe_pdr(acc_base, acc_var_all),
        })

        for vt, gv in var.groupby("variation_type"):
            acc_vt = gv["correct"].mean()
            out.append({
                "model": model,
                "method": method,
                "acc_base": acc_base,
                "acc_var_all": acc_var_all,
                "pdr_all_%": safe_pdr(acc_base, acc_var_all),
                "n_base": int(len(base)),
                "n_var": int(len(gv)),
                "variation_type": vt,
                "acc_var": acc_vt,
                "pdr_%": safe_pdr(acc_base, acc_vt),
            })

    return pd.DataFrame(out).sort_values(["model","method","variation_type"])

In [ ]:
def debug_one_cot(backend, row, max_new_tokens=512, temperature=0.2, top_p=0.95):
    msgs = build_prompt(row["question"])
    print("\n" + "="*120)
    print("ID:", row["id"])
    print("split:", row["split"], "| variation_type:", row["variation_type"], "| base_id:", row["base_id"])
    print("-"*120)
    print("PROMPT:")
    for m in msgs:
        print(f"[{m['role']}]\n{m['content']}\n")

    raw = backend.generate(msgs, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p)
    pred = extract_final(raw)
    ok = is_correct(pred, row["answer"])

    print("-"*120)
    print("RAW OUTPUT:\n", raw)
    print("-"*120)
    print("PARSED FINAL:", pred)
    print("GOLD:", row["answer"])
    print("CORRECT:", ok)

debug_one_cot(deepseek_r1_qwen7b, pilot_df.iloc[0])
debug_one_cot(qwen3_4b, pilot_df.iloc[0])
debug_one_cot(qwen3_06b, pilot_df.iloc[0])
debug_one_cot(qwen3_17b, pilot_df.iloc[0])


ID: gsm100_0027__clean__v1
split: base | variation_type: base | base_id: gsm100_0027
------------------------------------------------------------------------------------------------------------------------
PROMPT:
[system]
Solve the math problem carefully.
Briefly explain with 1-3 sentences.
End with exactly one line in the format:
FINAL: <answer>


[user]
Ruth goes to school 8 hours a day and 5 days a week. She is in math class 25% of this time. How many hours per week does she spend in math class?

------------------------------------------------------------------------------------------------------------------------
RAW OUTPUT:
 Okay, so Ruth goes to school 8 hours a day, and she goes there 5 days a week. I need to figure out how many hours she spends in math class each week. The problem says she's in math class 25% of the time. 

First, I'll calculate the total number of hours she spends in school each week. That should be 8 hours/day multiplied by 5 days/week. Let me do that math

In [ ]:
def debug_one_sc(backend, row, sc_n=3, max_new_tokens=128, temperature=0.8, top_p=0.95):
    msgs = build_prompt(row["question"])
    print("\n" + "="*120)
    print("ID:", row["id"])
    print("split:", row["split"], "| variation_type:", row["variation_type"], "| base_id:", row["base_id"])
    print("-"*120)

    voted, counts, raw_samples = run_self_consistency(
        backend, msgs, n=sc_n, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p
    )
    ok = is_correct(voted, row["answer"])

    print("VOTE COUNTS:", counts)
    print("VOTED FINAL:", voted)
    print("GOLD:", row["answer"])
    print("CORRECT:", ok)
    print("-"*120)
    for i, raw in enumerate(raw_samples):
        print(f"[SAMPLE {i}] parsed:", extract_final(raw))
        print(raw[:600] + ("..." if len(raw) > 600 else ""))
        print("-"*120)

debug_one_sc(openthoughts, pilot_df.iloc[0], sc_n=3)

In [ ]:
tmp = evaluate_simple(
    llama32_3b,
    "Llama-3.2-3B-Instruct",
    df.sample(30, random_state=0),
    method="cot",
    max_new_tokens=512,
    temperature_cot=0.0
)
print("numeric answer rate:",
      tmp["pred"].str.match(r"^\s*-?\d+(\.\d+)?\s*$").mean())

print("accuracy:", tmp["correct"].mean())

In [ ]:
print("Empty pred rate:", (tmp["pred"].fillna("").str.strip()=="").mean())

In [ ]:
bad = tmp[tmp["correct"]==False].head(100)
bad[["id","variation_type","gold","pred","raw_response"]]

In [ ]:
MODEL_SPECS = [
    {
        "backend": openthoughts,
        "name": "OpenThinker-Agent-v1",
        "max_new_tokens_cot": 512,
        "max_new_tokens_sc": 512,
    },
    {
        "backend": qwen25_7b,
        "name": "Qwen2.5-7B-Instruct",
        "max_new_tokens_cot": 256,
        "max_new_tokens_sc": 256,
    },
    {
        "backend": qwen3_06b,
        "name": "Qwen3-0.6B",
        "max_new_tokens_cot": 512,
        "max_new_tokens_sc": 512,
    },
    {
        "backend": qwen3_17b,
        "name": "Qwen3-1.7B",
        "max_new_tokens_cot": 512,
        "max_new_tokens_sc": 512,
    },
    {
        "backend": qwen3_4b,
        "name": "Qwen3-4B",
        "max_new_tokens_cot": 512,
        "max_new_tokens_sc": 512,
    },
    {
        "backend": deepseek_r1_qwen7b,
        "name": "DeepSeek-R1-Distill-Qwen-7B",
        "max_new_tokens_cot": 512,
        "max_new_tokens_sc": 512,
    },
    {
        "backend": llama32_3b,
        "name": "Llama-3.2-3B-Instruct",
        "max_new_tokens_cot": 512,
        "max_new_tokens_sc": 512,
    },
]


In [ ]:
runs = []

for spec in MODEL_SPECS:
    backend = spec["backend"]
    name = spec["name"]

    # Full 300: baseline
    runs.append(
        evaluate_simple(
            backend, name, df,
            method="cot",
            max_new_tokens=spec["max_new_tokens_cot"],
            keep_raw=True
        )
    )

    # SC on subset only
    runs.append(
        evaluate_simple(
            backend, name, df_sc,
            method="sc", sc_n=3,
            max_new_tokens=spec["max_new_tokens_sc"],
            keep_raw=True
        )
    )

all_runs_df = pd.concat(runs, ignore_index=True)
display(summarize_pdr_clean(all_runs_df))

100%|██████████| 400/400 [3:36:20<00:00, 32.45s/it]


,model,method,acc_base,acc_var_all,pdr_all_%,n_base,n_var,variation_type,acc_var,pdr_%
0,DeepSeek-R1-Distill-Qwen-7B,cot,0.81,0.806000,0.493827,100,500,__ALL__,0.806000,0.493827
1,DeepSeek-R1-Distill-Qwen-7B,cot,0.81,0.806000,0.493827,100,100,critical_thinking,0.840000,-3.703704
2,DeepSeek-R1-Distill-Qwen-7B,cot,0.81,0.806000,0.493827,100,100,irrelevant_numeric,0.810000,0.000000
3,DeepSeek-R1-Distill-Qwen-7B,cot,0.81,0.806000,0.493827,100,100,rephrase,0.820000,-1.234568
4,DeepSeek-R1-Distill-Qwen-7B,cot,0.81,0.806000,0.493827,100,100,scale_tweak,0.820000,-1.234568
5,DeepSeek-R1-Distill-Qwen-7B,cot,0.81,0.806000,0.493827,100,100,symbolic_swap,0.740000,8.641975
6,DeepSeek-R1-Distill-Qwen-7B,sc,0.87,0.873333,-0.383142,100,300,__ALL__,0.873333,-0.383142
7,DeepSeek-R1-Distill-Qwen-7B,sc,0.87,0.873333,-0.383142,100,100,irrelevant_numeric,0.880000,-1.149425
8,DeepSeek-R1-Distill-Qwen-7B,sc,0.87,0.873333,-0.383142,100,100,scale_tweak,0.870000,0.000000
9,DeepSeek-R1-Distill-Qwen-7B,sc,0.87,0.873333,-0.383142,100,100,symbolic_swap,0.870000,0.000000


In [ ]:
runs = []

for spec in MODEL_SPECS:
    backend = spec["backend"]
    name = spec["name"]

    # Full 300: baseline
    runs.append(
        evaluate_simple(
            backend, name, df,
            method="cot",
            max_new_tokens=spec["max_new_tokens_cot"],
            keep_raw=True
        )
    )

    # SC on subset only
    runs.append(
        evaluate_simple(
            backend, name, df_sc,
            method="sc", sc_n=3,
            max_new_tokens=spec["max_new_tokens_sc"],
            keep_raw=True
        )
    )

all_runs_df = pd.concat(runs, ignore_index=True)
display(summarize_pdr_clean(all_runs_df))

100%|██████████| 400/400 [3:41:21<00:00, 33.20s/it]


,model,method,acc_base,acc_var_all,pdr_all_%,n_base,n_var,variation_type,acc_var,pdr_%
0,Qwen3-0.6B,cot,0.63,0.642000,-1.904762,100,500,__ALL__,0.642000,-1.904762
1,Qwen3-0.6B,cot,0.63,0.642000,-1.904762,100,100,critical_thinking,0.630000,0.000000
2,Qwen3-0.6B,cot,0.63,0.642000,-1.904762,100,100,irrelevant_numeric,0.670000,-6.349206
3,Qwen3-0.6B,cot,0.63,0.642000,-1.904762,100,100,rephrase,0.660000,-4.761905
4,Qwen3-0.6B,cot,0.63,0.642000,-1.904762,100,100,scale_tweak,0.580000,7.936508
5,Qwen3-0.6B,cot,0.63,0.642000,-1.904762,100,100,symbolic_swap,0.670000,-6.349206
6,Qwen3-0.6B,sc,0.73,0.696667,4.566210,100,300,__ALL__,0.696667,4.566210
7,Qwen3-0.6B,sc,0.73,0.696667,4.566210,100,100,irrelevant_numeric,0.750000,-2.739726
8,Qwen3-0.6B,sc,0.73,0.696667,4.566210,100,100,scale_tweak,0.660000,9.589041
9,Qwen3-0.6B,sc,0.73,0.696667,4.566210,100,100,symbolic_swap,0.680000,6.849315


In [ ]:
OUT_PREFIX = "/content/gsm_full"

wrong = all_runs_df[all_runs_df["correct"] == False].copy()
print("Wrong count:", len(wrong))
print("Wrong with rationale rate:", wrong["raw_response"].notna().mean())

wrong["gold_norm"] = wrong["gold"].map(normalize_answer)
wrong["pred_norm"] = wrong["pred"].map(normalize_answer)

wrong_path = f"{OUT_PREFIX}_wrong.jsonl"
wrong.to_json(wrong_path, orient="records", lines=True, force_ascii=False)

all_path = f"{OUT_PREFIX}_runs.jsonl"
all_runs_df.to_json(all_path, orient="records", lines=True, force_ascii=False)

wrong_summary = (wrong.groupby(["model","method","variation_type"])
                      .size()
                      .reset_index(name="wrong_count")
                      .sort_values("wrong_count", ascending=False))
display(wrong_summary)

wrong_summary_path = f"{OUT_PREFIX}_wrong_summary.csv"
wrong_summary.to_csv(wrong_summary_path, index=False)

examples = (wrong.groupby(["model","method","variation_type"], group_keys=False)
                 .head(3)[["model","method","variation_type","id","question","gold","pred","raw_response"]])
examples_path = f"{OUT_PREFIX}_wrong_examples_top3.jsonl"
examples.to_json(examples_path, orient="records", lines=True, force_ascii=False)

print("Saved:")
print(" -", wrong_path)
print(" -", all_path)
print(" -", wrong_summary_path)
print(" -", examples_path)

Wrong count: 308
Wrong with rationale rate: 1.0


,model,method,variation_type,wrong_count
5,DeepSeek-R1-Distill-Qwen-7B,cot,symbolic_swap,26
0,DeepSeek-R1-Distill-Qwen-7B,cot,base,19
2,DeepSeek-R1-Distill-Qwen-7B,cot,irrelevant_numeric,19
3,DeepSeek-R1-Distill-Qwen-7B,cot,rephrase,18
15,Qwen3-4B,cot,symbolic_swap,18
4,DeepSeek-R1-Distill-Qwen-7B,cot,scale_tweak,18
11,Qwen3-4B,cot,critical_thinking,17
1,DeepSeek-R1-Distill-Qwen-7B,cot,critical_thinking,16
13,Qwen3-4B,cot,rephrase,15
10,Qwen3-4B,cot,base,15


Saved:
 - /content/gsm_full_wrong.jsonl
 - /content/gsm_full_runs.jsonl
 - /content/gsm_full_wrong_summary.csv
 - /content/gsm_full_wrong_examples_top3.jsonl
